# 03. Mortality model

Train **TabPFN-3.5** as a binary classifier for `MORTSTAT`.

Mortality-linkage fields are excluded from `X` to prevent leakage. Chronological age (`RIDAGEYR`) is intentionally retained here because mortality prediction is a different task.

In [1]:
from pathlib import Path
import sys
import json
import time
import numpy as np
import pandas as pd

root = Path.cwd().resolve()
for candidate in [root, *root.parents]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        root = candidate
        break
sys.path.insert(0, str(root))

from src.config import PROCESSED_DIR, MODELS_DIR, RESULTS_DIR, TARGET_MORTALITY, RANDOM_STATE
from src.data_utils import (load_json, apply_categorical_dtypes)
from src.evaluation import classification_metrics
from src.model_utils import make_mortality_model, save_model

train = pd.read_csv(PROCESSED_DIR / "train_mortality.csv")
test = pd.read_csv(PROCESSED_DIR / "test_mortality.csv")
metadata = load_json(PROCESSED_DIR / "metadata.json")
features = metadata["mortality_features"]

X_train = train[features].copy()
y_train = pd.to_numeric(train[TARGET_MORTALITY], errors="coerce").astype(int)
X_test = test[features].copy()
y_test = pd.to_numeric(test[TARGET_MORTALITY], errors="coerce").astype(int)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train mortality prevalence:", y_train.mean())

Train: (8225, 72) Test: (2057, 72)
Train mortality prevalence: 0.3181762917933131


In [2]:
# Categorical features are explicitly declared for TabPFN-3.5.
categorical_features = metadata["categorical_mortality_features"]

X_train_tabpfn = apply_categorical_dtypes(X_train, categorical_features)
X_test_tabpfn = apply_categorical_dtypes(X_test, categorical_features)

print("Categorical features used by TabPFN-3.5:", categorical_features)
print(X_train_tabpfn[categorical_features].dtypes)

Categorical features used by TabPFN-3.5: ['RIAGENDR', 'RIDRETH1', 'DMDEDUC2', 'DMDMARTL', 'SMQ040', 'PAQ180']
RIAGENDR    category
RIDRETH1    category
DMDEDUC2    category
DMDMARTL    category
SMQ040      category
PAQ180      category
dtype: object


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

baselines = {
    "LogisticRegression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "RandomForest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
}

rows = []
for name, model in baselines.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    metrics = classification_metrics(y_test, prob, pred)
    metrics.update({"Model": name, "Seconds": time.time() - t0})
    rows.append(metrics)

baseline_results = pd.DataFrame(rows).set_index("Model")
baseline_results

,AUROC,AUPRC,Brier,BalancedAccuracy,F1,Accuracy,Seconds
Model,,,,,,,
LogisticRegression,0.923420,0.879418,0.106524,0.848811,0.780899,0.848323,0.154508
RandomForest,0.919881,0.871869,0.104082,0.844551,0.788064,0.865338,1.277491


In [6]:
# TabPFN-3.5 classification.
mortality_model = make_mortality_model(thinking_effort="high")
t0 = time.time()
mortality_model.fit(X_train_tabpfn, y_train)
mort_prob = mortality_model.predict_proba(X_test_tabpfn)[:, 1]
mort_pred = mortality_model.predict(X_test_tabpfn)
mort_seconds = time.time() - t0

tabpfn_mortality_metrics = classification_metrics(y_test, mort_prob, mort_pred)
tabpfn_mortality_metrics.update({"Model": "TabPFN-3.5", "Seconds": mort_seconds})
print(tabpfn_mortality_metrics)

{'AUROC': 0.9273171731174569, 'AUPRC': 0.8931452567051672, 'Brier': 0.08915564090780113, 'BalancedAccuracy': 0.8413709373317553, 'F1': 0.7928513403736799, 'Accuracy': 0.8760330578512396, 'Model': 'TabPFN-3.5', 'Seconds': 121.94998121261597}


In [7]:
all_results = pd.concat([baseline_results.reset_index(), pd.DataFrame([tabpfn_mortality_metrics])], ignore_index=True)
all_results = all_results[["Model", "AUROC", "AUPRC", "Brier", "BalancedAccuracy", "F1", "Accuracy", "Seconds"]]
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results.to_csv(RESULTS_DIR / "mortality_model_comparison.csv", index=False)

pred_df = pd.DataFrame({
    "ROW_ID": test["ROW_ID"],
    "RIDAGEYR": test["RIDAGEYR"],
    "MORTSTAT": y_test,
    "MORTALITY_PROB_TABPFN": mort_prob,
    "MORTALITY_PRED_TABPFN": mort_pred,
})
pred_df.to_csv(RESULTS_DIR / "mortality_test_predictions.csv", index=False)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_record = save_model(mortality_model, MODELS_DIR / "mortality_model.json")
with open(MODELS_DIR / "mortality_model_record.json", "w", encoding="utf-8") as fh:
    json.dump(model_record, fh, indent=2)

print(all_results)

                Model     AUROC     AUPRC     Brier  BalancedAccuracy  \
0  LogisticRegression  0.923420  0.879418  0.106524          0.848811   
1        RandomForest  0.919881  0.871869  0.104082          0.844551   
2          TabPFN-3.5  0.927317  0.893145  0.089156          0.841371   

         F1  Accuracy     Seconds  
0  0.780899  0.848323    0.154508  
1  0.788064  0.865338    1.277491  
2  0.792851  0.876033  121.949981  
